# TechNova Laptops: Agentic Customer Service Chatbot

A LangGraph multi-agent chatbot using the **Google AI (Gemini) API**:
- **Router agent**: classifies each message as ORDER / REFUND / PRODUCT / SMALLTALK
- **Order Status Agent**: looks up orders and changes their quantity (tools)
- **Refund Policy Agent**: quotes the policy and checks if an order is eligible for a refund (tools)
- **Product Advisor Agent**: recommends laptops by budget, use case and shipping time (tools)
- **MemorySaver**: saves the conversation by `thread_id`, so later turns remember earlier ones

**Before running:** open the 🔑 *Secrets* panel on the left, add `GOOGLE_API_KEY` (from https://aistudio.google.com/apikey), and turn on *Notebook access*.

In [ ]:
# 1. Get the code + data and install dependencies
!git clone https://github.com/minchree/Customer-Service-Chatbot.git
%cd Customer-Service-Chatbot
!pip install -q -r requirements.txt

In [ ]:
# 2. Build the chatbot. The API key is read from Colab Secrets, never hard-coded
from chatbot import CustomerServiceChatbot, build_model, run_demo

bot = CustomerServiceChatbot(build_model())
print("Chatbot ready, model:", bot.model.model)

In [ ]:
# 3. Visualize the agent graph: router -> one of 4 nodes -> END
from IPython.display import Image
Image(bot.graph.get_graph().draw_mermaid_png())

## Demo: two multi-turn conversations
Shows all three agent types and memory across 6 turns each (order ID, name, budget and needs are remembered). The transcript is also saved to `outputs/demo_transcript.txt`.

In [ ]:
run_demo(bot)

## How the memory works
MemorySaver stores a checkpoint of the graph state after every turn. Here we read back what it saved for one thread.

In [ ]:
import uuid
thread = str(uuid.uuid4())
for msg in ["Hi, I'm Sam and my order is ORD-7311.",
            "When was it delivered?",
            "Can I still return it?"]:
    route, reply = bot.chat(msg, thread)
    print(f"USER: {msg}
  [{route}]
NOVA: {reply}
")

# Inspect the saved state for this thread
saved = bot.graph.get_state({"configurable": {"thread_id": thread}})
print("Messages stored by MemorySaver:", len(saved.values["messages"]))
for m in saved.values["messages"]:
    print(f"- {m.type:5} | {str(m.content)[:80]}")

## Try it yourself
Type messages to chat. Type `quit` to stop.

In [ ]:
thread = str(uuid.uuid4())
while (msg := input("YOU: ").strip()).lower() not in {"quit", "exit"}:
    route, reply = bot.chat(msg, thread)
    print(f"  [{route}]
NOVA: {reply}
")